# SpliceBERT v2 — Does a frozen pre-mRNA LM encode a splice site's **partner**?**One notebook. Real hg38 sequence, real GENCODE splice sites. No synthetic constructs.**### Why v2 exists — v1 was fatally confounded| flaw | evidence ||---|---|| **Positional label leak** | `dataset.py` set `acceptor = left_flank`, a *constant*. The true acceptor sat at position **200 in 100.0%** of examples. || **Geometry oracle** | Both endpoints were pinned to the window edges. A model reading **zero nucleotides** scored **PMR 1.000**. || **Window length leaked the exon length** | `len(seq) = 400 + exon_len` exactly → length-only adversary 0.495. || **Task solvable by site detection** | Decoys were *spurious* AG/GT motifs, so an **oracle site detector scored PMR 1.000** — partner binding was never tested. || **Readout couldn't express binding** | `score = f(h_a) + g(h_d) + L(len)` — no joint term. |### What v2 changes1. **Decoys are REAL annotated splice sites** — donors that exist in the same gene but that this acceptor **never** pairs with in *any* isoform. Site detection is therefore useless *by construction*: an oracle site detector sits exactly at chance.2. **Fixed 1024nt window**, filled with **real genome** (never PAD — pad-count is a length channel) → the length channel carries **0 bits**.3. **Acceptor offset randomised** over a range **independent of exon length** → no positional or length leak.4. **Candidates permuted** by a seed derived from the *sequence*, never the label → no index leak.5. **Readout that can express binding** (bilinear), with the additive readout kept as a **falsification control** that *must* fail.### The identified stratumThe headline is the **`adversarial`** stratum: examples where **both** the nearest-downstream rule **and** the exon-length prior prefer a *decoy*. There:| rival | accuracy ||---|---|| length prior | **0.000** (below chance by construction) || nearest-downstream | **0.000** (below chance by construction) || oracle site detector | **= chance** (every candidate is a real site) |**Any model above chance there must be using partner information.** That is the whole experiment.

## 0 · Setup

In [ ]:
import os, sys, time, json!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no GPU"!pip -q install transformers pyfaidx --upgrade 2>/dev/null | tail -1from google.colab import drive; drive.mount('/content/drive')DRIVE = '/content/drive/MyDrive/splicebert_v2'os.makedirs(DRIVE, exist_ok=True)# Code is version-controlled, not a blob: clone the pinned branch.REPO_URL = 'https://github.com/cam-berger/gfm-circuit-discovery.git'BRANCH   = 'splicebert-v2-partner-binding'if not os.path.exists('/content/gfm'):    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/gfmsys.path.insert(0, '/content/gfm/src')!cd /content/gfm && git log --oneline -1import torchDEV = 'cuda' if torch.cuda.is_available() else 'cpu'print('device:', DEV, torch.cuda.get_device_name(0) if DEV=='cuda' else '')

## 1 · Resources — genome, annotation, frozen SpliceBERT

In [ ]:
import globGTF   = f'{DRIVE}/gencode.v50.basic.annotation.gtf.gz'FASTA = f'{DRIVE}/hg38.fa'# --- SpliceBERT: find it wherever it already lives on Drive (do not re-upload 400MB).# The v1 run left it under splicebert_run/raw_data/models/... ; search there first,# then fall back to a recursive glob over the whole Drive.CANDIDATES = [    f'{DRIVE}/SpliceBERT.1024nt',    '/content/drive/MyDrive/splicebert_run/raw_data/models/splicebert_official_archive/extracted/models/SpliceBERT.1024nt',]MODEL = next((c for c in CANDIDATES if os.path.isdir(c)), None)if MODEL is None:    hits = glob.glob('/content/drive/MyDrive/**/SpliceBERT.1024nt', recursive=True)    MODEL = hits[0] if hits else Noneassert MODEL and os.path.isfile(f'{MODEL}/config.json'), (    'SpliceBERT.1024nt not found on Drive. Searched:\n  ' + '\n  '.join(CANDIDATES) +    '\n  /content/drive/MyDrive/**/SpliceBERT.1024nt\n'    'Set MODEL = "<path to the SpliceBERT.1024nt folder>" by hand.')print('model:', MODEL)# genome + annotation (downloaded once, cached on Drive)if not os.path.exists(GTF):    !wget -q -O {GTF} https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.basic.annotation.gtf.gzif not os.path.exists(FASTA):    !wget -q -O {FASTA}.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz && gunzip -f {FASTA}.gzprint('resources OK')

## 2 · Build the v2 dataset (real sites, leak-free geometry)Cached to Drive — a Colab disconnect never costs you this.

In [ ]:
from gfmcd.splicing.dataset import PyfaidxGenomefrom gfmcd.splicing.partner_data import (PartnerConfig, build_partner_dataset,    split_partner_examples, save_partner_jsonl, load_partner_jsonl, LengthPrior, stratify)CACHE = f'{DRIVE}/partner_dataset.jsonl'cfg = PartnerConfig()      # window 1024, acceptor offset U[100,300], real-genome fillercfg.validate()if os.path.exists(CACHE):    X = load_partner_jsonl(CACHE); print(f'loaded cache: {len(X):,}')else:    t=time.time()    X = build_partner_dataset(GTF, PyfaidxGenome(FASTA), cfg)   # FULL GENOME    save_partner_jsonl(X, CACHE)    print(f'built {len(X):,} examples in {time.time()-t:.0f}s -> {CACHE}')S = split_partner_examples(X)print({k: len(v) for k,v in S.items()}, '(test = chr8,chr9 SEALED)')

## 3 · 🚨 LEAK GUARDRAILS — hard gateIf any of these fail, **stop**. v1 shipped because nobody ran them.

In [ ]:
from collections import Counterimport numpy as nppos = Counter(x.acceptor_pos for x in X)seqlens = {len(x.sequence) for x in X}tix = Counter(x.true_index for x in X)checks = [ ('acceptor position is not constant',  len(pos) >= 50,               f'{len(pos)} distinct (v1 had 1)'), ('no single acceptor position dominates', max(pos.values())/len(X) < 0.10, f'max share {max(pos.values())/len(X):.1%} (v1: 100%)'), ('window length carries 0 bits',       len(seqlens) == 1,            f'len(seq) in {seqlens}'), ('true index is not always 0',         tix[0]/len(X) < 0.75,         f'index-0 share {tix[0]/len(X):.1%}'), ('every example is a forced choice',   all(len(x.candidates) >= 2 for x in X), 'min 2 candidates'),]print(f"{'guardrail':42s} {'':4s} detail")ok = Truefor name, passed, detail in checks:    ok &= passed    print(f"{name:42s} {'PASS' if passed else 'FAIL':4s} {detail}")assert ok, 'LEAK GUARDRAIL FAILED — do not proceed'print('\nAll leak guardrails pass.')

## 4 · Strata`adversarial` = both the nearest-downstream rule *and* the length prior prefer a decoy. That is the identified stratum.

In [ ]:
prior = LengthPrior([x.true_exon_length for x in S['train']])   # fit on TRAIN onlyfor x in X: x.metadata['stratum'] = stratify(x, prior)strata = Counter(x.metadata['stratum'] for x in X)print(strata)VAL = S['val']V_adv = [x for x in VAL if x.metadata['stratum']=='adversarial']print(f"val: {len(VAL):,}   val-adversarial: {len(V_adv):,}")

## 5 · 🚨 RIVAL TABLE — the pre-registered shortcut controls**The gate:** on `adversarial`, the oracle site detector must be *at* chance and the distance/length rivals *below* it. Only then is the task identified.

In [ ]:
from gfmcd.splicing.rivals import (PositionOnlyRival, GeometryOracleRival, LengthPriorRival,    NearestDownstreamRival, OracleSiteDetectorRival, CandidateIndexRival, top1, chance_level)mu_d   = float(np.mean([x.true_donor_pos for x in S['train']]))mu_end = float(np.mean([len(x.sequence)-x.true_donor_pos for x in S['train']]))RIVALS = [PositionOnlyRival(mu_d), GeometryOracleRival(mu_end), LengthPriorRival(prior),          NearestDownstreamRival(), OracleSiteDetectorRival(), CandidateIndexRival()]def table(exs, title):    ch = chance_level(exs)    print(f"\n{title}  (n={len(exs):,}, chance={ch:.3f})")    print(f"  {'rival':24s} {'shortcut':26s} {'acc':>6s}")    for r in RIVALS:        a = top1(r, exs)        flag = '  <-- ABOVE CHANCE' if a > ch + 0.05 else ''        print(f"  {r.name:24s} {r.shortcut:26s} {a:6.3f}{flag}")    return chch_all = table(VAL,   'ALL val')ch_adv = table(V_adv, 'ADVERSARIAL val  <-- the identified stratum')osd = top1(OracleSiteDetectorRival(), V_adv)assert abs(osd - ch_adv) < 1e-6, 'oracle site detector is NOT at chance — dataset is broken'print(f"\nGATE PASS: perfect site detection buys nothing ({osd:.3f} == chance {ch_adv:.3f}).")print(f"Any model above {ch_adv:.3f} on this stratum MUST be using partner information.")

## 6 · Precompute frozen embeddings  ⚡The backbone is frozen and tiny, so embed **once** and cache only the hidden states at the site positions. Every probe then trains on the cache in seconds. This is the H100/L4 optimisation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLMtok = AutoTokenizer.from_pretrained(MODEL)bb  = AutoModelForMaskedLM.from_pretrained(MODEL, output_hidden_states=True).to(DEV).eval()for p in bb.parameters(): p.requires_grad_(False)NL = bb.config.num_hidden_layersprint(f'SpliceBERT: {NL} layers x {bb.config.num_attention_heads} heads, hidden {bb.config.hidden_size}')EMB = f'{DRIVE}/emb_layer{NL}.npz'MAXC = max(len(x.candidates) for x in X)@torch.no_grad()def embed(exs, bs=64):    A = np.zeros((len(exs), bb.config.hidden_size), dtype=np.float16)    Dn= np.zeros((len(exs), MAXC, bb.config.hidden_size), dtype=np.float16)    M = np.zeros((len(exs), MAXC), dtype=bool)    for i in range(0, len(exs), bs):        ch = exs[i:i+bs]        enc = tok([' '.join(x.sequence) for x in ch], return_tensors='pt', padding=True).to(DEV)        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(DEV=='cuda')):            h = bb(**enc).hidden_states[-1].float()          # [B,S,D]        for j,x in enumerate(ch):            A[i+j] = h[j, x.acceptor_pos+1].cpu().numpy()     # +1 for [CLS]            for k,d in enumerate(x.candidates):                Dn[i+j,k] = h[j, d+1].cpu().numpy(); M[i+j,k] = True        if i % (bs*40) == 0: print(f'  {i}/{len(exs)}', end='\r')    return A, Dn, Mif os.path.exists(EMB):    z=np.load(EMB); print('loaded embedding cache')else:    t=time.time(); parts={}    for name in ['train','val']:        a,d,m = embed(S[name]); parts[f'{name}_a']=a; parts[f'{name}_d']=d; parts[f'{name}_m']=m    np.savez_compressed(EMB, **parts); z=np.load(EMB)    print(f'embedded in {time.time()-t:.0f}s -> {EMB}')

## 7 · Arm 1 — partner binding**Bilinear** (can express binding) vs **Additive control** (cannot — *must* be at chance) vs **AcceptorQuery** (reads `h_a` + offset, never `h_d`).

In [ ]:
from gfmcd.splicing.partner_readout import (CandidateBatch, BilinearPartnerReadout,    AdditiveDonorReadout, AcceptorQueryReadout, loss_fn, top1 as t1)def mkbatch(exs, a, d, m):    off = np.zeros(m.shape, dtype=np.float32); lab = np.zeros(len(exs), dtype=np.int64)    for i,x in enumerate(exs):        for k,c in enumerate(x.candidates): off[i,k] = c - x.acceptor_pos        lab[i] = x.true_index    return CandidateBatch(torch.tensor(a,dtype=torch.float32), torch.tensor(d,dtype=torch.float32),                          torch.tensor(off), torch.tensor(m), torch.tensor(lab))TR = mkbatch(S['train'], z['train_a'], z['train_d'], z['train_m'])VA = mkbatch(S['val'],   z['val_a'],   z['val_d'],   z['val_m'])adv_idx = torch.tensor([i for i,x in enumerate(S['val']) if x.metadata['stratum']=='adversarial'])def train(cls, seed, epochs=30, bs=256, lr=1e-3):    torch.manual_seed(seed)    m = cls(bb.config.hidden_size).to(DEV); opt = torch.optim.AdamW(m.parameters(), lr=lr)    n = TR.labels.shape[0]    for _ in range(epochs):        perm = torch.randperm(n)        for i in range(0, n, bs):            j = perm[i:i+bs]            b = CandidateBatch(TR.acceptor[j],TR.donor[j],TR.offset[j],TR.mask[j],TR.labels[j]).to(DEV)            opt.zero_grad(); loss_fn(m(b), b).backward(); opt.step()    m.eval()    with torch.no_grad():        v = VA.to(DEV); lg = m(v)        all_acc = t1(lg, v).item()        av = CandidateBatch(VA.acceptor[adv_idx],VA.donor[adv_idx],VA.offset[adv_idx],                            VA.mask[adv_idx],VA.labels[adv_idx]).to(DEV)        adv_acc = t1(m(av), av).item()    return all_acc, adv_accRES={}for name, cls in [('bilinear (hypothesis)', BilinearPartnerReadout),                  ('ADDITIVE CONTROL', AdditiveDonorReadout),                  ('acceptor-query', AcceptorQueryReadout)]:    r=[train(cls,s) for s in range(3)]    RES[name]=(np.mean([x[0] for x in r]), np.std([x[0] for x in r]),               np.mean([x[1] for x in r]), np.std([x[1] for x in r]))    print(f'{name:24s} all={RES[name][0]:.3f}+-{RES[name][1]:.3f}  ADVERSARIAL={RES[name][2]:.3f}+-{RES[name][3]:.3f}')

## 8 · Verdict

In [ ]:
print(f"{'model / rival':26s} {'ALL':>14s} {'ADVERSARIAL':>14s}")print('-'*58)print(f"{'chance':26s} {ch_all:14.3f} {ch_adv:14.3f}")for r in RIVALS:    print(f"{r.name:26s} {top1(r,VAL):14.3f} {top1(r,V_adv):14.3f}")print('-'*58)for k,(a,sa,d,sd) in RES.items():    print(f"{k:26s} {a:14.3f} {d:14.3f}")bil = RES['bilinear (hypothesis)'][2]; add = RES['ADDITIVE CONTROL'][2]print('\n' + '='*58)if add > ch_adv + 0.05:    print('❌ CONTROL FAILED: the additive readout beats chance on the adversarial')    print('   stratum. It CANNOT express partner binding, so a shortcut remains.')    print('   DO NOT report the bilinear number. Find the leak.')elif bil > ch_adv + 0.05:    print('✅ PARTNER BINDING DETECTED.')    print(f'   bilinear {bil:.3f} > chance {ch_adv:.3f}, additive control at chance ({add:.3f}).')    print('   Site detection, position, length, distance and list order are all')    print('   neutralised by construction -> only partner information remains.')    print('   => proceed to the causal/tracing stage.')else:    print('⚪ NULL RESULT (publishable).')    print(f'   bilinear {bil:.3f} ~ chance {ch_adv:.3f}: SpliceBERT\'s frozen representation')    print('   does NOT encode partner identity beyond site-hood, length and distance.')    print('   v1\'s apparent "partner selection" was a positional leak + site detection.')print('='*58)json.dump({'chance':{'all':ch_all,'adversarial':ch_adv},           'rivals':{r.name:{'all':top1(r,VAL),'adversarial':top1(r,V_adv)} for r in RIVALS},           'models':{k:{'all':v[0],'all_sd':v[1],'adversarial':v[2],'adversarial_sd':v[3]} for k,v in RES.items()},           'n':{'all':len(VAL),'adversarial':len(V_adv)}},          open(f'{DRIVE}/arm1_results.json','w'), indent=2)print(f'\nsaved -> {DRIVE}/arm1_results.json')

## 9 · Arm 2 — causal (gated)**Only run if Arm 1 passed.** Does the acceptor's representation *read* its partner? Mask the true donor's tokens: the partner score should collapse. Mask a matched real *decoy* donor: it should not.

In [ ]:
@torch.no_grad()def masked_acc(model, exs, which='true', bs=32):    hits=0    for i in range(0,len(exs),bs):        ch=exs[i:i+bs]; seqs=[]        for x in ch:            s=list(x.sequence)            tgt = x.true_donor_pos if which=='true' else (x.real_decoy_donor_pos[0] if x.real_decoy_donor_pos else x.true_donor_pos)            for p in range(max(0,tgt-2), min(len(s),tgt+2)): s[p]='N'            seqs.append(' '.join(s))        enc=tok(seqs,return_tensors='pt',padding=True).to(DEV)        with torch.autocast('cuda',dtype=torch.bfloat16,enabled=(DEV=='cuda')):            h=bb(**enc).hidden_states[-1].float()        A=torch.stack([h[j,x.acceptor_pos+1] for j,x in enumerate(ch)])        C_=max(len(x.candidates) for x in ch)        D=torch.zeros(len(ch),C_,h.shape[-1],device=DEV); M=torch.zeros(len(ch),C_,dtype=torch.bool,device=DEV)        O=torch.zeros(len(ch),C_,device=DEV); L=torch.zeros(len(ch),dtype=torch.long,device=DEV)        for j,x in enumerate(ch):            for k,c in enumerate(x.candidates):                D[j,k]=h[j,c+1]; M[j,k]=True; O[j,k]=c-x.acceptor_pos            L[j]=x.true_index        b=CandidateBatch(A,D,O,M,L)        hits += (model(b).argmax(-1)==L).sum().item()    return hits/len(exs)if bil > ch_adv + 0.05 and add <= ch_adv + 0.05:    torch.manual_seed(0)    m = BilinearPartnerReadout(bb.config.hidden_size).to(DEV)    # retrain quickly on the cache    opt=torch.optim.AdamW(m.parameters(),lr=1e-3); n=TR.labels.shape[0]    for _ in range(30):        perm=torch.randperm(n)        for i in range(0,n,256):            j=perm[i:i+256]; b=CandidateBatch(TR.acceptor[j],TR.donor[j],TR.offset[j],TR.mask[j],TR.labels[j]).to(DEV)            opt.zero_grad(); loss_fn(m(b),b).backward(); opt.step()    m.eval()    base = masked_acc(m, V_adv, 'none') if False else None    a_true  = masked_acc(m, V_adv, 'true')    a_decoy = masked_acc(m, V_adv, 'decoy')    print(f'adversarial stratum, bilinear readout:')    print(f'  clean                 : {bil:.3f}')    print(f'  TRUE donor masked     : {a_true:.3f}   <- should COLLAPSE toward chance ({ch_adv:.3f})')    print(f'  DECOY donor masked    : {a_decoy:.3f}   <- should NOT collapse (specificity)')    json.dump({'clean':bil,'true_masked':a_true,'decoy_masked':a_decoy,'chance':ch_adv},              open(f'{DRIVE}/arm2_causal.json','w'), indent=2)else:    print('Arm 1 did not pass its gate — Arm 2 skipped by design.')